## Run the fire expansion and merging algorithm

`Fire_Forward` is responsible for reading in the preprocessed data created in the Ingest notebook

In [1]:
import datetime
import pandas as pd
import geopandas as gpd

from fireatlas import FireMain, FireTime, FireObj, postprocess
from fireatlas.utils import timed

tst = [2017, 12, 25, 'AM']
ted = [2018, 1, 30, 'AM']
region = ('Thomas_fire_no_newyear_with_long_check', [-119.68162521,   34.2798998 , -118.92518097,   34.63745683])
list_of_ts = list(FireTime.t_generator(tst, ted))


# tst = [2017, 12, 1, 'AM']
# ted = [2017, 12, 6, 'AM']
# region = ('Thomas_fire_test', [-119.68162521,   34.2798998 , -118.92518097,   34.63745683])

2025-06-26 20:12:29,867 - fireatlas.FireLog - INFO - logger initialized!


Since we want to use precisely the files that we just created in the Ingest notebook. We will set the `read_location` to "local".

In [2]:
allfires, allpixels, t_saved = FireMain.Fire_Forward(tst=tst, ted=ted, restart=False, region=region, read_location="local")

2025-06-26 20:12:30,753 - fireatlas.FireLog - WARNING - No saved version of allfires and allpixels
2025-06-26 20:12:30,760 - fireatlas.FireLog - INFO - func:read_preprocessed took: 6.46 ms
2025-06-26 20:12:30,764 - fireatlas.FireLog - INFO - func:read_preprocessed took: 3.34 ms
2025-06-26 20:12:30,768 - fireatlas.FireLog - INFO - func:read_preprocessed took: 3.52 ms
2025-06-26 20:12:30,771 - fireatlas.FireLog - INFO - func:read_preprocessed took: 3.00 ms
2025-06-26 20:12:30,775 - fireatlas.FireLog - INFO - func:read_preprocessed took: 3.27 ms
2025-06-26 20:12:30,779 - fireatlas.FireLog - INFO - func:read_preprocessed took: 2.93 ms
2025-06-26 20:12:30,782 - fireatlas.FireLog - INFO - func:read_preprocessed took: 3.18 ms
2025-06-26 20:12:30,786 - fireatlas.FireLog - INFO - func:read_preprocessed took: 3.13 ms
2025-06-26 20:12:30,789 - fireatlas.FireLog - INFO - func:read_preprocessed took: 3.14 ms
2025-06-26 20:12:30,793 - fireatlas.FireLog - INFO - func:read_preprocessed took: 3.23 ms
2

# Concepts
This outputs two dataframes: an `allpixels` dataframe with 1 row per pixel and an `allfires` geodataframe with one row per-fire/per-t. 

The core concept is that if you use a dataframe to back the allfires and fire objects there are well-defined ways to serialize that to disk whenever you like (aka no more pickles!).

Here's a bit of an overview of the lifecycle of each of these dataframes:

## allpixels:

- At the start of `Fire_Forward` all of the preprocessed pixel data is loaded and concatenated into one long dataframe.
- Each row represents a fire pixel and there is a unique id per row.
- As `Fire_Forward` iterates through the timesteps of interest the `allpixels` dataframe is updated in place.
- Each `Fire` object refers to the `allpixels` object as the source of truth and does not hold pixel data but instead refers to subsets of the `allpixels` dataframe to return `n_pixels` or `newpixels`.
- Merging fires at a particular `t` can update the `allpixels` at a former timestep.
- When `Fire_Forward` is complete, the `allpixels` object can be serialized to csv (or any tabular format) optionally partitioned into files by `t`.
- This dataframe can be used:
    - together with `allfires_gdf` to rehydrate the `allfires` object at the latest `t` in order to run `Fire_Forward` on one new ingest file.
    - independently to write the `nplist` output file for largefires

## allfires_gdf:

- At the start of `Fire_Forward` a new geodataframe object is initialized. It has a column for each of the `Fire` attributes that take a non-trivial amount of time to compute (`ftype`, `hull`, `fline`...).
- As `Fire_Forward` iterates through the timesteps of interest it writes a row for every fire that is burning (aka has new pixels) at the `t`.
- So each row contains the information about one fire at one `t`. The index is a MultiIndex of `(fid, t)`
- Merging fires at a particular `t` updates the `mergeid` on the existing rows (_this part I am not totally confident is correct_).
- When `Fire_Forward` is complete, the `allfires_gdf` object can be serialized to geoparquet (this is the best choice since it contains multiple geometry columns) optionally partitioned into files by `t`.
- This geodataframe can be used:
    - together with `allpixels` to rehydrate the `allfires` object at the latest `t` in order to run `Fire_Forward` on one new ingest file.
    - independently to write all the snapshot and largefires output files.

Side note: I like that in this branch the `allpixels` dataframe is referenced by all the `Fire` objects but it isn't copied around. This is different from how it works in `preprocess` where each `Fire` object (at each `t`) has its own dataframe. It is also different than the original version of this algorithm where each `Fire` object (at each `t`) holds a bunch of lists.

## Serialize to disk

By default the allfires_gdf and allpixels will be serialized to disk as part of `Fire_Forward`

- allpixels -> one file for each t (one row for each pixel).
- allfires -> one geoparquet file to hold all information about each fire at each time (one row for each burning fire at each t).

## Read from disk

In [2]:
allpixels = postprocess.read_allpixels(tst, ted, region, location="local")

NameError: name 'postprocess' is not defined

In [5]:
allfires_gdf = postprocess.read_allfires_gdf(tst, ted, region, location="local")

2025-06-26 18:36:15,491 - fireatlas.FireLog - INFO - func:read_allfires_gdf took: 25.69 ms


## Pick out the large fires

Let's compare the existing object-oriented approach with the new geodataframe approach

In [7]:
%%time
from fireatlas import FireGpkg_sfs

large_fires_original = FireGpkg_sfs.find_largefires(allfires)

CPU times: user 500 µs, sys: 0 ns, total: 500 µs
Wall time: 413 µs


In [8]:
large_fires_new = postprocess.find_largefires(allfires_gdf)

2025-06-23 19:48:26,188 - fireatlas.FireLog - INFO - func:find_largefires took: 4.75 ms


In [9]:
assert set(large_fires_original) == set(large_fires_new), "The large fires should match"

## Rehydrate the latest allfires

This is pretty experimental, but at least in theory you should be able to rehydrate the allfires object based on the allfires_gdf. If this works as expected it would let you pick up from a particular t and run another step of `Fire_Forward`.

This should be equivalent to the allfires object that we generated at the top of this notebook.

In [ ]:
a = FireObj.Allfires.rehydrate(tst, ted, region, include_dead=False, read_location="local")
a